# Oil Price Forecasting & Market Intelligence (WTI)

This notebook builds an applied, explainable ML system for WTI price forecasting. It covers data ingestion, feature engineering, model training, evaluation, forecast generation with uncertainty, and a transparent signal engine.

**Data scope:** WTI price series from project data.
**Goal:** Produce credible, reproducible forecasts and interpretable market signals.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10
})

data_dir = Path("..") / "data"
prices_raw: pd.DataFrame = pd.read_csv(data_dir / "oil_prices.csv")


def clean_prices(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    if "date" not in df.columns and "year" in df.columns:
        df["date"] = pd.to_datetime(df["year"].astype(str) + "-12-31", errors="coerce")
    if "price" not in df.columns:
        if "oil_price" in df.columns:
            df = df.rename(columns={"oil_price": "price"})
        elif "value" in df.columns:
            df = df.rename(columns={"value": "price"})
    df = df.loc[:, ["date", "price"]]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")
    df["price"] = df["price"].interpolate(limit_direction="both").ffill().bfill()
    return df


prices: pd.DataFrame = clean_prices(prices_raw)
prices = prices.set_index("date").asfreq("D")
prices["price"] = prices["price"].interpolate("time").ffill().bfill()
prices = prices.reset_index()

display(prices.head())

## 1. Data Overview

We validate date coverage, missing values, and price distribution before modeling. This establishes data quality and the effective sample window.

In [ ]:
summary = pd.DataFrame({
    "metric": ["rows", "start_date", "end_date", "missing_price"],
    "value": [
        len(prices),
        prices["date"].min().date() if len(prices) else None,
        prices["date"].max().date() if len(prices) else None,
        int(prices["price"].isna().sum())
    ]
})

display(summary)

fig, ax = plt.subplots()
ax.plot(prices["date"], prices["price"], color="#1f4e79", linewidth=2)
ax.set_title("WTI Price Series")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

We build lagged values, rolling statistics, momentum, percent change, and z-scores for anomaly context.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["lag_1"] = data["price"].shift(1)
    data["lag_3"] = data["price"].shift(3)
    data["lag_7"] = data["price"].shift(7)
    data["rolling_mean_7"] = data["price"].rolling(7).mean()
    data["rolling_mean_30"] = data["price"].rolling(30).mean()
    data["rolling_std_7"] = data["price"].rolling(7).std()
    data["momentum_7"] = data["price"].pct_change(7)
    data["momentum_30"] = data["price"].pct_change(30)
    data["pct_change"] = data["price"].pct_change()
    rolling_mean = data["pct_change"].rolling(30).mean()
    rolling_std = data["pct_change"].rolling(30).std()
    data["z_score"] = (data["pct_change"] - rolling_mean) / rolling_std
    data = data.dropna().reset_index(drop=True)
    return data


features = build_features(prices)
feature_cols = [
    "lag_1",
    "lag_3",
    "lag_7",
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_std_7",
    "momentum_7",
    "momentum_30",
    "pct_change",
    "z_score"
]

features.head()

## 3. Train/Test Split

We preserve time order by training on earlier observations and testing on the most recent segment.

In [ ]:
split_idx = int(len(features) * 0.8)
train = features.iloc[:split_idx].copy()
test = features.iloc[split_idx:].copy()

X_train = train[feature_cols]
y_train = train["price"]
X_test = test[feature_cols]
y_test = test["price"]

display((train.shape, test.shape))

fig, ax = plt.subplots()
ax.plot(train["date"], train["price"], color="#1f4e79", label="Train")
ax.plot(test["date"], test["price"], color="#f28e2b", label="Test")
ax.set_title("Train/Test Split")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Model Comparison

We train a naive baseline, linear regression, ARIMA, and a tree model to compare performance on held-out data.

In [ ]:
def metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred)))
    }


results = []

naive_pred = test["lag_1"].to_numpy()
results.append({"model": "naive", **metrics(y_test, naive_pred)})

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
results.append({"model": "linear", **metrics(y_test, linear_pred)})

train_series = train.set_index("date")["price"]
test_series = test.set_index("date")["price"]
arima_fit = ARIMA(train_series, order=(1, 1, 1)).fit()
arima_pred = arima_fit.forecast(steps=len(test_series))
results.append({"model": "arima", **metrics(test_series.to_numpy(), arima_pred)})

gbr_model = GradientBoostingRegressor(random_state=42)
gbr_model.fit(X_train, y_train)
gbr_pred = gbr_model.predict(X_test)
results.append({"model": "gbr", **metrics(y_test, gbr_pred)})

results_df: pd.DataFrame = pd.DataFrame(results)
display(results_df)

## 5. Evaluation Metrics

MAE and RMSE provide an honest comparison across models. We select the model with the lowest RMSE.

In [ ]:
results_df = results_df.sort_values("rmse").reset_index(drop=True)
best_model = str(results_df.loc[0, "model"])

display(results_df)

display(best_model)

## 6. Forecast Visualization

We generate 7-day and 30-day forecasts with confidence intervals using the best-performing model.

In [ ]:
def prediction_interval(pred, residuals):
    sigma = float(np.nanstd(residuals)) if len(residuals) else 0.0
    lower = pred - 1.96 * sigma
    upper = pred + 1.96 * sigma
    return lower, upper


def recursive_forecast(model, history_df, steps):
    history = history_df.copy()
    last_date = history["date"].iloc[-1]
    future_dates = pd.date_range(last_date, periods=steps + 1, inclusive="right")
    preds = []
    for date in future_dates:
        extended = pd.concat(
            [history, pd.DataFrame({"date": [date], "price": [np.nan]})],
            ignore_index=True,
        )
        feat = build_features(extended)
        X_last = feat.iloc[-1][feature_cols].to_numpy().reshape(1, -1)
        pred = float(model.predict(X_last)[0])
        preds.append(pred)
        history = pd.concat(
            [history, pd.DataFrame({"date": [date], "price": [pred]})],
            ignore_index=True,
        )
    return future_dates, np.array(preds)


def arima_forecast(series, steps):
    fit = ARIMA(series, order=(1, 1, 1)).fit()
    forecast_res = fit.get_forecast(steps=steps)
    pred = forecast_res.predicted_mean.to_numpy()
    conf = forecast_res.conf_int(alpha=0.05)
    lower = conf.iloc[:, 0].to_numpy()
    upper = conf.iloc[:, 1].to_numpy()
    return pred, lower, upper


full_series = prices.set_index("date")["price"]
residuals_map = {
    "naive": y_test.to_numpy() - naive_pred,
    "linear": y_test.to_numpy() - linear_pred,
    "arima": test_series.to_numpy() - np.array(arima_pred),
    "gbr": y_test.to_numpy() - gbr_pred
}

if best_model == "naive":
    forecast_30 = np.repeat(float(full_series.iloc[-1]), 30)
    lower_30, upper_30 = prediction_interval(forecast_30, residuals_map["naive"])
elif best_model == "linear":
    _, forecast_30 = recursive_forecast(linear_model, prices, 30)
    lower_30, upper_30 = prediction_interval(forecast_30, residuals_map["linear"])
elif best_model == "arima":
    forecast_30, lower_30, upper_30 = arima_forecast(full_series, 30)
else:
    _, forecast_30 = recursive_forecast(gbr_model, prices, 30)
    lower_30, upper_30 = prediction_interval(forecast_30, residuals_map["gbr"])

forecast_7 = forecast_30[:7]
lower_7 = lower_30[:7]
upper_7 = upper_30[:7]

future_dates_30 = pd.date_range(full_series.index[-1], periods=31, inclusive="right")

fig, ax = plt.subplots()
ax.plot(full_series.tail(180).index, full_series.tail(180).values, color="#1f4e79", label="Actual")
ax.plot(future_dates_30, forecast_30, color="#f28e2b", label="Forecast")
ax.fill_between(future_dates_30, lower_30, upper_30, color="#f28e2b", alpha=0.2, label="CI")
ax.set_title("Forecast vs Recent Actuals")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

def direction_from_forecast(values):
    change = (values[-1] - values[0]) / max(abs(values[0]), 1e-6)
    if change >= 0.01:
        return "bullish"
    if change <= -0.01:
        return "bearish"
    return "neutral"

best_row = results_df.loc[results_df["model"] == best_model].head(1)
best_mae = float(best_row["mae"].to_numpy()[0])
best_rmse = float(best_row["rmse"].to_numpy()[0])

forecast_payload = {
    "model_used": best_model,
    "mae": best_mae,
    "rmse": best_rmse,
    "forecast_7": forecast_7.tolist(),
    "forecast_30": forecast_30.tolist(),
    "confidence_interval_30": [
        {"lower": float(l), "upper": float(u)} for l, u in zip(lower_30, upper_30)
    ],
    "direction": direction_from_forecast(forecast_30)
}

display(forecast_payload)

## 7. Interpretation

We surface feature importance for explainable models and highlight what drives the forecast.

In [ ]:
feature_importance = None

if best_model == "linear":
    importance = np.abs(linear_model.coef_) * X_train.std().to_numpy()
    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": importance
    }).sort_values("importance", ascending=False)
elif best_model == "gbr":
    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": gbr_model.feature_importances_
    }).sort_values("importance", ascending=False)

if feature_importance is not None:
    display(feature_importance.head(8))

## 8. Signal Engine

We blend forecast direction, momentum, and volatility regime into a transparent signal.

In [ ]:
returns = prices["price"].pct_change()
rolling_vol = returns.rolling(30).std()
vol_percentile = float(rolling_vol.rank(pct=True).iloc[-1] * 100)

if vol_percentile < 25:
    vol_regime = "low"
elif vol_percentile < 50:
    vol_regime = "medium"
elif vol_percentile < 75:
    vol_regime = "high"
else:
    vol_regime = "extreme"

momentum_30 = float(prices["price"].pct_change(30).iloc[-1])
forecast_direction = forecast_payload["direction"]

momentum_strength = min(abs(momentum_30) * 6, 1.0)
vol_penalty = {"low": 0.1, "medium": 0.25, "high": 0.45, "extreme": 0.65}[vol_regime]

signal_strength = 0.45 * momentum_strength + 0.35 * (1 - vol_penalty) + 0.2 * 0.6
signal_strength = float(min(max(signal_strength, 0.05), 0.95))

signal_confidence = 0.5 * (1 - vol_penalty) + 0.3 * (1 - min(abs(momentum_30) * 4, 1.0)) + 0.2 * 0.6
signal_confidence = float(min(max(signal_confidence, 0.05), 0.95))

signal_payload = {
    "direction": forecast_direction,
    "strength": signal_strength,
    "confidence": signal_confidence,
    "volatility_regime": vol_regime,
    "reasoning": [
        f"Forecast direction is {forecast_direction}",
        f"30-day momentum is {momentum_30:.2%}",
        f"Volatility regime is {vol_regime}"
    ]
}

signal_payload

## 9. Limitations

- The model set is intentionally simple; it does not encode macro drivers or structural breaks.
- ARIMA and regression assume stability in historical dynamics and may understate regime shifts.
- Interpolated daily data is a modeling convenience, not a substitute for higher-frequency observations.

In [ ]:
model_output = {
    "model_used": forecast_payload["model_used"],
    "mae": forecast_payload["mae"],
    "rmse": forecast_payload["rmse"],
    "forecast": forecast_payload["forecast_30"],
    "confidence_interval": forecast_payload["confidence_interval_30"],
    "direction": forecast_payload["direction"]
}

display(model_output)

## 10. Final Insights

- The best-performing model is selected using time-aware evaluation with MAE and RMSE.
- Forecasts include uncertainty bands to avoid overconfidence.
- Signals combine momentum and volatility context to provide explainable market posture.

## 11. Next Steps

- Add macro covariates (inventories, dollar index, refinery utilization) to improve predictive power.
- Validate models with a rolling backtest and expand to Brent for cross-benchmark robustness.
- Operationalize retraining and monitoring in the backend services.